# Exercise 13: The tune diagram

**Block 4: Resonances**

**Learning objectives**
* Draw resonance lines $m Q_x + n Q_y = p$ in the ($Q_x$, $Q_y$) plane and know their order $|m| + |n|$.
* Scan thousands of working points at once and find, by tracking, which resonance lines a given
  magnet really excites.
* Make an amplitude-vs-tune map and read off the **dynamic aperture** as a function of the tune.

**Motivation.** Choosing the tunes (the *working point*) is one of the first decisions for any ring.
The tune diagram is the map of "dangerous places". Here you make one yourself, the way it is done for
real machines: by tracking particles and measuring how regular their motion is.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()
BETA = 4.0

## 1. Resonance lines

A resonance occurs when $m Q_x + n Q_y = p$ for integers $m$, $n$, $p$: after a few turns the particle
returns to the same phases in both planes, and small kicks add up. The **order** is $|m| + |n|$. Low
orders are the most dangerous. Lines with odd $n$ (dashed below) need skew fields (or a vertical offset)
to be driven.

`P.tune_diagram(ax, max_order)` draws all lines up to a given order.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))
P.tune_diagram(axs[0], 3, (0, 1), (0, 1)); axs[0].legend(loc="upper right", fontsize=8)
axs[0].set_title("orders 1 to 3")
P.tune_diagram(axs[1], 5, (0, 0.5), (0, 0.5)); axs[1].legend(loc="upper right", fontsize=8)
axs[1].set_title("orders 1 to 5, zoom on [0, 0.5]")
plt.show()

## 2. Scanning working points

We use the Hénon-like ring again, but with a twist: every particle gets **its own** tunes ($Q_x$, $Q_y$)
(`L.HenonRing` accepts arrays of tunes). So with one tracking run we test a whole grid of working points.
Each particle starts at the same amplitude $\hat{x} = \hat{y} = A$.

For each particle we measure the tune in the first and second half of the run (`D.chaos_indicator`):
regular motion gives the same tune twice (indicator $\approx$ −8 or lower), chaotic motion gives a wandering tune
(indicator $\approx$ −3 … −1). Lost particles get `np.nan`.

**Task.** Complete `scan_tunes`.

In [ ]:
def scan_tunes(qx, qy, amplitude, knl=(), ksl=(), n_turns=1024):
    """Chaos indicator and lost turn for particles at (qx[i], qy[i]) starting at x_hat = y_hat = amplitude.

    Returns (indicator, lost_turn): arrays of the same length as qx; indicator is nan for lost particles.
    """
    ring = L.HenonRing(qx, qy, BETA, 0.0, BETA, 0.0, knl=knl, ksl=ksl)
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
# small check on a linear ring: all particles regular, none lost
q = np.array([0.11, 0.23, 0.31])
ind, lost = scan_tunes(q, q[::-1], 5e-3, knl=[], n_turns=512)
assert np.all(ind < -8) and np.all(lost == 512)
print("GOOD! scan_tunes works")

**Predict first.** We put a single **normal sextupole** ($k_2L = 10 \text{m}^{-2}$) in the ring. Which resonance
lines do you expect to see in the scan? (Exercise 1: a normal sextupole kicks $p_x \approx x^2 − y^2$ and
$p_y \approx xy$.)

*Your prediction:*

In [ ]:
# (about 10 s) 100 x 100 working points, 1024 turns each
n = 100
QX, QY = np.meshgrid(np.linspace(0.01, 0.5, n), np.linspace(0.01, 0.5, n))
qx, qy = QX.ravel(), QY.ravel()
ind_s, lost_s = scan_tunes(qx, qy, 4e-3, knl=[0, 0, 10.0])
print(f"{np.isnan(ind_s).sum()} of {len(qx)} working points lose the particle")

In [ ]:
def show_scan(ind, title, max_order=4):
    fig, axs = plt.subplots(1, 2, figsize=(14, 6.3))
    for ax, lines in zip(axs, [False, True]):
        im = ax.pcolormesh(QX, QY, ind.reshape(QX.shape), cmap="plasma", vmin=-9, vmax=-2, shading="auto")
        ax.pcolormesh(QX, QY, np.where(np.isnan(ind), 1.0, np.nan).reshape(QX.shape), cmap="Greys",
                      vmin=0, vmax=1, shading="auto")
        if lines:
            P.tune_diagram(ax, max_order, (0.01, 0.5), (0.01, 0.5), alpha=0.6, lw_scale=1.5)
            ax.legend(loc="upper right", fontsize=7)
        ax.set_xlim(0.01, 0.5); ax.set_ylim(0.01, 0.5); ax.set_aspect("equal")
        ax.set_xlabel(r"$Q_x$"); ax.set_ylabel(r"$Q_y$")
    plt.colorbar(im, ax=axs, label=r"chaos indicator $\log_10|\Delta Q|$ (black: lost)", shrink=0.8)
    axs[0].set_title(title); axs[1].set_title("with resonance lines up to order %d" % max_order)
    plt.show()

show_scan(ind_s, "normal sextupole, A = 4 √(mm mrad)")

Black bands are working points where the particle is lost. You should recognise:
* $Q_x = 1/3$ (vertical band): the third-order resonance $3Q_x = 1$ of Exercise 11;
* $Q_x + 2Q_y = 1$ (the diagonal band from (0, 0.5) to (0.5, 0.25)): a *coupled* third-order resonance,
  driven by the $y^2$ term in $\Delta p_x$ and the $xy$ term in $\Delta p_y$;
* the diagonal $Q_x = Q_y$ ($2Q_x − 2Q_y = 0$) and other fourth-order lines: a sextupole drives them at
  *second* order (two kicks combined), so they are chaotic but rarely lose the particle;
* thinner, fainter lines: $Q_x − 2Q_y = 0$ and higher orders.

**Task.** Look at the plot and decide: is the line $3Q_y = 1$ (horizontal line at $Q_y = 1/3$) strongly
excited? Why or why not? Write your answer below.

*Your answer:*

## 3. Adding an octupole

**Predict first.** Now add an octupole ($k_3L = 1000 \text{m}^{-3}$). Which new lines appear?

*Your prediction:*

In [ ]:
# (about 10 s)
ind_so, lost_so = scan_tunes(qx, qy, 4e-3, knl=[0, 0, 10.0, 1000.0])
show_scan(ind_so, "sextupole + octupole, A = 4 √(mm mrad)")

The octupole adds fourth-order lines: $4Q_x = 1$ ($Q_x = 0.25$), $4Q_y = 1$, $2Q_x \pm 2Q_y = p$.
It also makes the tune depend on amplitude, which shifts and widens the bands: the particle's actual tune
is not the "bare" tune on the axes (Exercise 14).

## 4. Dynamic aperture vs tune

Now we keep $Q_y$ fixed and scan $Q_x$ and the starting amplitude (horizontal only, $y = 0$). For each
($Q_x$, amplitude) we track 1000 turns and record how many turns the particle survived.

**Task.** Complete the scan and, for each $Q_x$, find the dynamic aperture: the smallest amplitude
whose particle is lost (`np.nan` if none).

In [ ]:
# (about 3 s)
qx1 = np.linspace(0.15, 0.45, 151)
amps = np.linspace(0.25e-3, 25e-3, 100)
QXa, AMP = np.meshgrid(qx1, amps)
ring_a = L.HenonRing(QXa.ravel(), 0.38, BETA, 0.0, BETA, 0.0, knl=[0, 0, 10.0, 1000.0])
bn = np.zeros((4, QXa.size)); bn[0, :] = AMP.ravel()
res_a = T.track(ring_a.denormalise(bn), ring_a.one_turn_map, 1000, store=False)
surv_turns = res_a.lost_turn.reshape(QXa.shape)

# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.pcolormesh(qx1, amps * MM, np.log10(surv_turns), cmap="viridis", shading="auto")
ax.plot(qx1, da * MM, color="w", lw=2, label="dynamic aperture")
for q, lab in [(1 / 3, "1/3"), (1 / 4, "1/4"), (1 / 5, "1/5"), (2 / 5, "2/5"), (2 / 7, "2/7")]:
    ax.axvline(q, color="w", ls=":", lw=1)
    ax.text(q, amps[-1] * MM * 1.01, lab, ha="center", va="bottom")
plt.colorbar(im, ax=ax, label=r"$\log_10$(turns survived)")
ax.set_xlabel(r"$Q_x$"); ax.set_ylabel(r"starting amplitude $\hat x$ [$\sqrt{\mu\text{m}}$]")
ax.legend(loc="upper left")
plt.show()
assert np.nanargmin(da) == np.argmin(np.abs(qx1 - 1 / 3)) or abs(qx1[np.nanargmin(da)] - 1 / 3) < 0.02, \
    "the DA should be smallest near Qx = 1/3"
print(f"smallest DA {np.nanmin(da) * MM:.2f} at Qx = {qx1[np.nanargmin(da)]:.3f}; "
      f"largest {np.nanmax(da) * MM:.2f} at Qx = {qx1[np.nanargmax(da)]:.3f}")

The dynamic aperture collapses near $Q_x = 1/3$ (sextupole, third order) and dips near 1/4 (octupole and
sextupole at second order) and other low-order fractions. Note that the dips are not exactly at the
fractions: the octupole shifts the tune with amplitude, so the resonance is met at a different bare tune.

## Reflection questions

1. If you had to choose a working point for this ring, where would you put it? Why?
2. Why are the bands wider (and the DA lower) at larger amplitude $A$? Try $A$ = 2 and 6 $\sqrt{\mu\text{m}}$.
3. Real machines also have errors in every magnet. How would that change the tune diagram?